#列の削除により多重共線性の解消を目指す

In [1]:
#import
import pandas as pd
from sklearn.model_selection import train_test_split  #データの分割

from sklearn.preprocessing import StandardScaler #標準化
from sklearn.model_selection import KFold #K分割交差検証
from sklearn.model_selection import cross_validate  #K分割交差検証

from sklearn.linear_model import LinearRegression #回帰
from sklearn.preprocessing import PolynomialFeatures  #交互作用特徴量
from sklearn.linear_model import Ridge  #リッジ回帰
from sklearn.linear_model import Lasso  #ラッソ回帰
from sklearn.tree import DecisionTreeRegressor  #回帰木

from statsmodels.stats.outliers_influence import variance_inflation_factor  #VIF
from sklearn.decomposition import PCA  #PCA

In [2]:
#データフレームの読み込み
'''
df1 VIFの高さを解消するために一部の列を消去する前のデータフレーム
df2 〃した後のデータフレーム

以降は基本的にdf2を用いて分析を行う。
sc_x,df_yはdf2を基に作成する。
ただし、必要があればdf1も利用する。
'''
df2 = pd.read_csv('datafiles/df2_after_drop.csv')

sc_x = df2.drop(['SalePrice'], axis = 1)
df_y = pd.DataFrame(df2['SalePrice'])

In [3]:
print(df2.columns)

Index(['MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual', 'OverallCond',
       'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtUnfSF',
       ...
       'Condition2_Norm', 'Condition2_PosA', 'Condition2_PosN',
       'Condition2_RRAe', 'Condition2_RRAn', 'Condition2_RRNn',
       'GarageFinish_NA', 'GarageFinish_RFn', 'GarageFinish_Unf', 'SalePrice'],
      dtype='object', length=250)


In [4]:
#K分割交差検証のための準備
kf=KFold(n_splits=5, shuffle=True, random_state=0)

In [5]:
#各種モデルの作成

#重回帰
model1 = LinearRegression()
model1.fit(sc_x, df_y)

#リッジ回帰
model2 = Ridge(alpha = 100)
model2.fit(sc_x, df_y)

#ラッソ回帰
model3 = Lasso(alpha = 100)
model3.fit(sc_x, df_y)

#回帰木
model4 = DecisionTreeRegressor(max_depth = 10, random_state = 0)
model4.fit(sc_x, df_y)

,criterion,'squared_error'
,splitter,'best'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


In [6]:
#依然として多重共線性は高いため、今後も多重共線性の解消を目指す。

#vifを見る
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x, i) for i in range(sc_x.shape[1])]
vif_df["features"]=sc_x.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 100]
display(df_high_vif)


,VIF_Factor,features
22,2178.616803,GarageYrBlt
56,960.199106,MiscFeature_NA
58,802.650747,MiscFeature_Shed
71,105.805972,Exterior2nd_VinylSd
124,115.837730,Exterior1st_VinylSd
149,233.699047,GarageQual_TA
163,283.746702,GarageCond_TA
183,165.010225,RoofStyle_Gable
185,153.477931,RoofStyle_Hip
246,2202.752816,GarageFinish_NA


In [7]:
#一列dropしてみる
sc_x = sc_x.drop(['GarageFinish_NA'], axis = 1)
#vifを見る
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x, i) for i in range(sc_x.shape[1])]
vif_df["features"]=sc_x.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 100]
display(df_high_vif)

model2.fit(sc_x, df_y)
result = cross_validate(model2, sc_x, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

,VIF_Factor,features
56,959.675823,MiscFeature_NA
58,802.110365,MiscFeature_Shed
71,105.785392,Exterior2nd_VinylSd
124,115.837134,Exterior1st_VinylSd
149,233.384665,GarageQual_TA
163,282.411676,GarageCond_TA
183,164.851995,RoofStyle_Gable
185,153.344617,RoofStyle_Hip


完成したmodel2のスコア＝0.7788228561257453


In [8]:
#一列dropしてみる
sc_x = sc_x.drop(['MiscFeature_NA'], axis = 1)
#vifを見る
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x, i) for i in range(sc_x.shape[1])]
vif_df["features"]=sc_x.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 100]
display(df_high_vif)

model2.fit(sc_x, df_y)
result = cross_validate(model2, sc_x, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

,VIF_Factor,features
70,105.733960,Exterior2nd_VinylSd
123,115.678525,Exterior1st_VinylSd
148,233.363856,GarageQual_TA
162,282.329291,GarageCond_TA
182,164.851570,RoofStyle_Gable
184,153.344550,RoofStyle_Hip


完成したmodel2のスコア＝0.7788175817571734


In [9]:
#3列dropしてみる
sc_x = sc_x.drop(['Exterior2nd_VinylSd', 'GarageCond_TA', 'RoofStyle_Gable'], axis = 1)
#vifを見る
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x, i) for i in range(sc_x.shape[1])]
vif_df["features"]=sc_x.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 100]
display(df_high_vif)

#スコアの確認
model2.fit(sc_x, df_y)
result = cross_validate(model2, sc_x, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

,VIF_Factor,features


完成したmodel2のスコア＝0.7785744518848134


In [10]:
#vifを見る
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x, i) for i in range(sc_x.shape[1])]
vif_df["features"]=sc_x.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 10]

df_high_vif.sort_values('VIF_Factor', ascending=False)

,VIF_Factor,features
128,90.503234,ExterCond_TA
151,81.381629,GarageType_Attchd
126,76.363708,ExterCond_Gd
173,74.401726,RoofMatl_CompShg
155,65.809905,GarageType_Detchd
147,54.715176,GarageQual_TA
22,53.575630,GarageYrBlt
49,51.800551,SaleType_New
118,51.486680,Exterior1st_MetalSd
76,49.162602,MSZoning_RL


In [11]:
#3列dropしてみる
sc_x = sc_x.drop(['ExterCond_TA', 'GarageType_Attchd'], axis = 1)
#vifを見る
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x, i) for i in range(sc_x.shape[1])]
vif_df["features"]=sc_x.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 10]
df_high_vif.sort_values('VIF_Factor', ascending=False)

,VIF_Factor,features
171,74.294750,RoofMatl_CompShg
146,54.692519,GarageQual_TA
49,51.798480,SaleType_New
118,51.119943,Exterior1st_MetalSd
197,49.071613,SaleCondition_Partial
76,49.044799,MSZoning_RL
10,43.948652,TotalBsmtSF
182,40.008059,Heating_GasA
65,39.335555,Exterior2nd_MetalSd
8,37.071897,BsmtFinSF1


In [12]:
#スコアの確認
model2.fit(sc_x, df_y)
result = cross_validate(model2, sc_x, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

完成したmodel2のスコア＝0.7786656098208722


In [13]:
#4列dropしてみる
sc_x = sc_x.drop(['RoofMatl_CompShg', 'GarageQual_TA', 'SaleType_New', 'Exterior1st_MetalSd'], axis = 1)
#vifを見る
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x, i) for i in range(sc_x.shape[1])]
vif_df["features"]=sc_x.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 10]
display(df_high_vif.sort_values('VIF_Factor', ascending=False))

,VIF_Factor,features
75,48.946791,MSZoning_RL
10,43.205565,TotalBsmtSF
178,39.953988,Heating_GasA
8,36.797813,BsmtFinSF1
9,35.658095,BsmtUnfSF
0,34.745525,MSSubClass
76,32.669914,MSZoning_RM
157,32.515975,MasVnrType_NA
114,28.835127,Exterior1st_CemntBd
156,28.079813,MasVnrType_BrkFace


In [14]:
result = cross_validate(model2, sc_x, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

完成したmodel2のスコア＝0.7786075560896861


In [15]:
#4列dropしてみる
sc_x = sc_x.drop(['MSZoning_RL', 'TotalBsmtSF', 'Heating_GasA', 'BsmtFinSF1'], axis = 1)
#vifを見る
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x, i) for i in range(sc_x.shape[1])]
vif_df["features"]=sc_x.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 10]
display(df_high_vif.sort_values('VIF_Factor', ascending=False))

,VIF_Factor,features
0,34.608125,MSSubClass
154,32.438191,MasVnrType_NA
111,28.758254,Exterior1st_CemntBd
153,28.017892,MasVnrType_BrkFace
59,27.951783,Exterior2nd_CmentBd
91,27.019800,FireplaceQu_NA
11,24.884128,GrLivArea
201,22.784311,Neighborhood_NAmes
117,21.672533,Exterior1st_VinylSd
184,18.709625,ExterQual_TA


In [16]:
#スコアの確認
model2.fit(sc_x, df_y)
result = cross_validate(model2, sc_x, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

完成したmodel2のスコア＝0.7960218940921336


In [17]:
#vifが大きいものを選択
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x, i) for i in range(sc_x.shape[1])]
vif_df["features"]=sc_x.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 28]

#4列dropしてみる
to_drop = set(df_high_vif['features'])
for c in to_drop:
    sc_x = sc_x.drop([c], axis = 1)

#vifを見る
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x, i) for i in range(sc_x.shape[1])]
vif_df["features"]=sc_x.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 10]
display(df_high_vif.sort_values('VIF_Factor', ascending=False))

#スコアの確認
model2.fit(sc_x, df_y)
result = cross_validate(model2, sc_x, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

,VIF_Factor,features
90,27.008619,FireplaceQu_NA
10,24.705651,GrLivArea
197,22.475783,Neighborhood_NAmes
115,20.689968,Exterior1st_VinylSd
180,18.680007,ExterQual_TA
202,18.112330,Neighborhood_OldTown
110,15.836130,Exterior1st_HdBoard
59,15.789927,Exterior2nd_HdBoard
8,15.459261,1stFlrSF
89,15.360314,FireplaceQu_Gd


完成したmodel2のスコア＝0.7950796667692217


In [18]:
#vifが大きいものを選択
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x, i) for i in range(sc_x.shape[1])]
vif_df["features"]=sc_x.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 18.5]

#4列dropしてみる
to_drop = set(df_high_vif['features'])
for c in to_drop:
    sc_x = sc_x.drop([c], axis = 1)

#vifを見る
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x, i) for i in range(sc_x.shape[1])]
vif_df["features"]=sc_x.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 10]
display(df_high_vif.sort_values('VIF_Factor', ascending=False))

#スコアの確認
model2.fit(sc_x, df_y)
result = cross_validate(model2, sc_x, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

,VIF_Factor,features
4,13.906200,YearBuilt
108,13.359459,Exterior1st_HdBoard
58,13.227892,Exterior2nd_HdBoard
217,11.707513,Condition2_Norm
50,11.399673,BsmtQual_TA
153,11.309360,BsmtFinType2_Unf
65,11.126744,Exterior2nd_Wd Sdng
113,10.833053,Exterior1st_Wd Sdng


完成したmodel2のスコア＝0.7971109145093968


In [19]:
#drop後のデータフレームを保存
df3_lowered_VIF = pd.concat([sc_x, df_y], axis = 1)
df3_lowered_VIF.to_csv('df3_lowered_VIF.csv', index=False)

In [20]:
#dropしてみる
sc_x = sc_x.drop(['YearBuilt', 'Exterior1st_HdBoard', 'Condition2_Norm', 'BsmtQual_TA'], axis = 1)
#vifを見る
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x, i) for i in range(sc_x.shape[1])]
vif_df["features"]=sc_x.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 10]
display(df_high_vif.sort_values('VIF_Factor', ascending=False))

#スコアの確認
model2.fit(sc_x, df_y)
result = cross_validate(model2, sc_x, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

,VIF_Factor,features
150,11.286445,BsmtFinType2_Unf


完成したmodel2のスコア＝0.7963257043346245


In [21]:
#dropしてみる
sc_x = sc_x.drop(['BsmtFinType2_Unf'], axis = 1)
#vifを見る
vif_df = pd.DataFrame()
vif_df["VIF_Factor"] = [variance_inflation_factor(sc_x, i) for i in range(sc_x.shape[1])]
vif_df["features"]=sc_x.columns
df_high_vif = vif_df.loc[vif_df['VIF_Factor'] >= 5]
display(df_high_vif.sort_values('VIF_Factor', ascending=False))

#スコアの確認
model2.fit(sc_x, df_y)
result = cross_validate(model2, sc_x, df_y, cv = kf, scoring = 'r2' , return_train_score = True)
print(f'完成したmodel2のスコア＝{sum(result['test_score'])/len(result['test_score'])}')

,VIF_Factor,features
101,9.790953,Functional_Typ
125,9.712928,KitchenQual_TA
63,8.190216,Exterior2nd_Wd Sdng
18,7.643792,GarageCars
110,7.642908,Exterior1st_Wd Sdng
202,7.536515,Foundation_PConc
124,7.334261,KitchenQual_Gd
19,7.019464,GarageArea
7,6.936213,1stFlrSF
79,6.676330,HouseStyle_1Story


完成したmodel2のスコア＝0.7965658597529963


In [22]:
#VIFの最大値が10以下になるよう特徴量を削除した場合、model2のスコアは最高で0.797となった。改変前のスコアは0.777であった。
#スコアが最高であったときの学習用データを保存した。